# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

This is a ranking / scoring problem, not classification. The output isn't a single yes/no per page — it's an ordered priority list, since a content team only has bandwidth to act on the top N pages each week. Precision@K (used throughout notebooks 01–02) is inherently a ranking metric, which confirms this framing over classification.

In [2]:
# Confirming: the pipeline's own metric (Precision@K) is a ranking metric, not a classification metric
def precision_at_k(scores, labels, k):
    import numpy as np
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

print("Precision@K measures rank quality (are the TOP K right), not just correct/incorrect per row.")
print("This confirms the ranking/scoring framing, not classification.")

Precision@K measures rank quality (are the TOP K right), not just correct/incorrect per row.
This confirms the ranking/scoring framing, not classification.


## 2. Target or proxy

Proxy target: is_declining_label, derived from trend_direction == "down". This is a proxy, not a perfect ground truth — it captures direction only, not magnitude, and comes from a single trailing window rather than a longer trend. I'm treating "declining" as the best available stand-in for "worth refreshing."

In [4]:
import pandas as pd
url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(df["trend_direction"].value_counts())
print(f"\nDeclining rate: {df['is_declining_label'].mean():.1%}")

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Declining rate: 54.2%


## 3. Success metric

Precision@K (K=20 and K=50) — of the top K pages my ranking flags, what fraction are actually declining. This fits the real action: a content team only touches a handful of pages per week, so recall across the whole 30,000-row dataset is irrelevant if the top 20–50 aren't right.

In [5]:
def precision_at_k(scores, labels, k):
    import numpy as np
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

print("Precision@K is the metric — see notebook 02 for hand-rule vs tree comparison using this exact function.")


Precision@K is the metric — see notebook 02 for hand-rule vs tree comparison using this exact function.


## 4. The unit of analysis, as a real dataframe

One row = one page. Each page is scored independently on staleness, visibility, and position signals to produce its refresh-priority rank.

In [6]:
unit_of_analysis = df[["days_since_last_update", "impressions_90d", "avg_position", "ctr", "word_count", "is_declining_label"]]
print(f"Unit of analysis: one row = one page. {unit_of_analysis.shape[0]} rows, {unit_of_analysis.shape[1]} columns")
unit_of_analysis.head(5)


Unit of analysis: one row = one page. 30000 rows, 6 columns


,days_since_last_update,impressions_90d,avg_position,ctr,word_count,is_declining_label
0,20,3803,10.6,0.76,3221.0,1
1,25,15320,20.3,0.05,2481.0,1
2,20,12581,36.5,0.09,3515.0,1
3,22,11751,6.2,0.49,NaN,0
4,14,19140,44.0,0.13,2803.0,1


## 5. Why ML beats a fixed rule here

Notebook 02 proved this empirically: a hand-written rule (stale × visible × impressions_90d) hits a ceiling because it can only combine signals linearly. The depth-2 decision tree found split points a human wouldn't guess by eye. And Discovery A showed the most "obvious" signal — search volume — has near-zero correlation with actual impressions, meaning intuition-built rules are actively misleading here, not just suboptimal.

In [7]:
corr = df["search_volume"].corr(df["impressions_90d"])
print(f"search_volume vs impressions_90d correlation: {corr:.3f}")
print("Near zero -> the 'obvious' signal a fixed rule would rely on doesn't actually predict the outcome.")


search_volume vs impressions_90d correlation: 0.001
Near zero -> the 'obvious' signal a fixed rule would rely on doesn't actually predict the outcome.


## Self-check

Before you submit, confirm each line honestly:

- [x] Named the ML task type (ranking/scoring, not classification)
- [x] Named the target/proxy and acknowledged its imperfection
- [x] Named the success metric (Precision@K) and why it fits the real action
- [x] Showed the unit of analysis as a real dataframe (one row = one page)
- [x] Explained why ML beats a fixed rule, backed by notebook 01/02 evidence
- [x] Committed to work/notebooks/ — then submit repo URL on the card